# Unconditional SE(3) Diffusion Training

Trains an **unconditional** SE(3) score network on a single protein MD
trajectory following the SinFusion single-trajectory protocol.

**What you get:** A model that generates new protein backbone conformations
from noise, sampling from the learned equilibrium distribution.

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive (persistent storage) |
| 2 | Clone repo & init submodules |
| 3 | Configure protein & training |
| 4 | Download & preprocess trajectory |
| 5 | Train |
| 6 | Generate samples |

## Step 1: Environment Setup

In [ ]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print('Drive mounted')

## Step 2: Clone Repository & Init Submodules

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/JiwonJJeong/winter-gen-pproject.git'

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run(['git', 'submodule', 'update', '--init', '--recursive'], check=True)
    # Symlink data/ to Drive for persistence
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print('data/ -> /content/drive/MyDrive/protein_data/data')
    print('Code ready')
else:
    assert os.path.isdir('gen_model'), 'Run from repo root'
    print(f'Local repo: {os.path.abspath(".")}')

## Step 3: Configuration

Edit the variables below for your protein and training preferences.

In [ ]:
# ---- Protein (single-trajectory, SinFusion-style) ----
PROTEIN   = '4o66_C'   # Protein name (as in atlas.csv)
REPLICA   = '1'        # Which replica to train on (1, 2, or 3)

# ---- Training ----
MAX_STEPS  = 200_000   # Total training steps
BATCH_SIZE = 8
LR         = 1e-4
GRAD_CLIP  = 1.0       # Gradient clipping (MDGen default)
EMA_DECAY  = 0.999     # EMA decay (0 = disabled)

# ---- Paths ----
DATA_DIR = './data'
SAVE_DIR = f'checkpoints/unconditional/{PROTEIN}'

print(f'Protein : {PROTEIN}_R{REPLICA}')
print(f'Steps   : {MAX_STEPS:,}')
print(f'Save to : {SAVE_DIR}')

## Step 4: Download & Prepare Data

In [ ]:
import os

if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError('data/ not linked to Drive. Run Step 1 first.')

!python scripts/download_and_prep.py {PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

traj_path = f'{DATA_DIR}/{PROTEIN}/{PROTEIN}_R{REPLICA}_latent.npy'
if os.path.exists(traj_path):
    import numpy as np
    arr = np.load(traj_path)
    print(f'Data ready: {traj_path}  shape={arr.shape}')
else:
    print(f'ERROR: {traj_path} not found')

## Step 5: Train

Runs `gen_model/train_unconditional.py` with all features:
EMA, gradient clipping, cosine LR, mixed precision.

In [ ]:
!python gen_model/train_unconditional.py \
    --protein {PROTEIN} \
    --replica {REPLICA} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --max_steps {MAX_STEPS} \
    --lr {LR} \
    --grad_clip {GRAD_CLIP} \
    --ema_decay {EMA_DECAY} \
    --save_dir {SAVE_DIR} \
    --num_workers 2

## Step 6: Generate Samples

Generate backbone conformations from noise using the trained model.

In [ ]:
import glob

ckpts = sorted(glob.glob(f'{SAVE_DIR}/uncond-*.ckpt'))
best_ckpt = ckpts[-1] if ckpts else f'{SAVE_DIR}/last.ckpt'
print(f'Checkpoint: {best_ckpt}')

!python gen_model/inference_unconditional.py \
    --checkpoint {best_ckpt} \
    --npy_path {DATA_DIR}/{PROTEIN}/{PROTEIN}_R{REPLICA}_latent.npy \
    --atlas_csv gen_model/splits/atlas.csv \
    --protein_name {PROTEIN} \
    --num_samples 100 \
    --num_steps 200 \
    --out_dir outputs/unconditional/{PROTEIN}

## Step 7: Visualize

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ca_path = f'outputs/unconditional/{PROTEIN}/ca_samples.npy'
if os.path.exists(ca_path):
    samples = np.load(ca_path)
    print(f'Generated: {samples.shape}')
    fig, axes = plt.subplots(1, min(5, len(samples)), figsize=(15, 3))
    if len(samples) == 1: axes = [axes]
    for i, ax in enumerate(axes):
        ca = samples[i]
        ax.plot(ca[:, 0], ca[:, 1], '-', lw=0.5)
        ax.set_title(f'Sample {i+1}')
        ax.set_aspect('equal'); ax.axis('off')
    plt.suptitle(f'{PROTEIN} — Generated CA traces')
    plt.tight_layout(); plt.show()
else:
    print('No samples found. Run inference first.')